# Twin runs of a deterministic LPJ-GUESS: how long does a small perturbation stay predictable?

Jakob Eisenhauer and Petran van Rens, Lund University, 2026. Code: MIT licence, data: CC BY 4.0 (see the end of this notebook).

This notebook states the problem, describes the data and reflects on FAIR. The analysis is in `01_short_kick_experiment.ipynb` and `02_long_rain_experiment.ipynb`.

## 1. The problem

Dynamic vegetation models such as LPJ-GUESS are run as ensembles and compared between scenarios. A difference between two runs is usually read as the effect of the scenario. That is only safe if the model forgets small differences in its state, or at least responds to them in a smooth, predictable way. If a 1 % nudge instead grows and sends the run onto another trajectory, part of every scenario difference is internal variability.

We test this at one tropical-African grid cell (P1400, about 1400 mm rain per year, forest-savanna band) with LPJ-GUESS made fully deterministic (cohort mode, one patch, constant climate, no random numbers), so every difference between runs comes from the dynamics.

Questions

1. Short experiment (notebook 01): after ten simultaneous small kicks (+-1 % of pools, density and forcing, +-0.1 K), for how many years can a linear model predict the response of the vegetation state from the kicks, and which kicks matter? (The perturbed initial conditions may not be reachable by the model itself through spin-up.)
2. Long experiment (notebook 02): after a change of about 1 % in one year of rain (so only the forcing is perturbed), do the runs separate from the control run?
3. In both: does fire (mode `C_fire_accumulate`) shorten or lengthen the predictable period compared with no fire (mode `A_nofire`)?

Methods from the course: regularisation (multi-task LASSO), cross-validation, bootstrap, hierarchical clustering. Why these, and their limits, is discussed in each notebook.

## 2. Workflow

1. LPJ-GUESS 4.1.1 with small patches (model half, not published) produces the data, `data/p1400_twin_runs_part1.csv` to `_part3.csv` (one table split in three files because of the upload size limit).
2. `01_short_kick_experiment.ipynb` reads the three csv files and writes `figures/state_*.png`.
3. `02_long_rain_experiment.ipynb` reads the three csv files and writes `figures/*NEE*.png`.

Everything after the csv files is in this folder and runs in about two minutes. To rerun all notebooks in place:

    jupyter nbconvert --to notebook --execute --inplace 0*.ipynb

## 3. Data description

The unit and meaning of every column are stored in `data/columns.csv`.

In [1]:
import pandas as pd

# the data is split in three files (upload size limit); they have the same columns and are joined row-wise
DATA_FILES = ["data/p1400_twin_runs_part1.csv", "data/p1400_twin_runs_part2.csv", "data/p1400_twin_runs_part3.csv"]
df = pd.concat([pd.read_csv(f) for f in DATA_FILES], ignore_index=True)
print(df.shape[0], "rows x", df.shape[1], "columns")

388826 rows x 24 columns


In [2]:
# runs (incl. control) and years per experiment and mode
df.groupby(["experiment", "mode"]).agg(runs=("member", "nunique"), first_t=("t", "min"), last_t=("t", "max"))

runs  first_t  last_t
experiment mode                                    
long_rain  A_nofire             31      -20    4579
           C_fire_accumulate    31      -20    4579
short_kick A_nofire            101        0     512
           C_fire_accumulate   101        0     512

In [3]:
# every column with unit and meaning (from data/columns.csv)
#pd.set_option("display.max_colwidth", 200)
#pd.read_csv("data/columns.csv").fillna("")

 How the csv was produced: LPJ-GUESS 4.1.1 with three small source patches (state/forcing kick; a deterministic fire rule `accumulate`: the yearly fire probability is summed and one full fire happens when the sum reaches 1; higher output precision). Cohort mode, 1 patch, `ifstochestab 0`, `ifstochmort 0`, `ifdisturb 0`, constant CO2 340 ppm and N deposition 2 kgN ha-1 yr-1, LPJ-GUESS demo climatology (monthly values repeated every year) at lon -6.25, lat 8.75. Spin-up of 2000 years once per mode, every run restarts from the saved state. Short experiment: Latin-hypercube design of the ten kicks (seed 7). Long experiment: eps = 0.01 * standard-normal draw (numpy `default_rng(42)`).

## 4. FAIR reflection

FAIR asks that data and the workflow around them are Findable, Accessible, Interoperable and Reusable. Below is what this project does for each letter and where it falls short.

Findable
- Unit and meaning of every column in `data/columns.csv`, title, authors, keywords and licence in `CITATION.cff`.
- The data is on GitHub, which gives no persistent identifier (DOI). Zenodo would be better: it gives a DOI and makes the metadata searchable.
- The notebooks are numbered in reading order, the README lists every file and the citation file names the model version (LPJ-GUESS 4.1.1).

Accessible
- One plain csv on GitHub, open licence, no login. On Zenodo it would stay available even if the file is withdrawn.
- Code and notebooks are plain text and run on free software (Python, packages pinned in `requirements.txt`).
- LPJ-GUESS is available from its developers (https://web.nateko.lu.se/lpj-guess/). With the stochastic processes frozen (as here) the dataset can also be generated with it.
- Short of FAIR: patches and run scripts (model half) are not published, and LPJ-GUESS has its own terms. So the csv can be analysed by anyone, but regenerating it needs the model and patches.

Interoperable
- UTF-8 csv in tidy (long) format, one row per run and year, with a unit and a description for every column in `data/columns.csv`.
- Variable names follow the LPJ-GUESS output files (`cmass`, `lai`, `cflux_NEE`, ...).
- Standard Jupyter notebooks and standard libraries (numpy, pandas, scipy, scikit-learn, matplotlib).

Reusable
- Clear licences (code MIT, data CC BY 4.0), model version, settings, patches, units for every column.
- Pinned package versions, seeded random numbers, and the settings of each analysis at the top of its notebook.
- Short of FAIR: the results describe one grid cell, one control run and one restart state. Other conclusions need new model runs.

## 5. Licence and citation

- Code (`plotlib.py`, notebooks): MIT licence, file `LICENSE`.
- Data (`data/p1400_twin_runs_part1.csv` to `_part3.csv`, `data/columns.csv`): CC BY 4.0, file `LICENSE-DATA.md`.
- Cite as in `CITATION.cff`. LPJ-GUESS: Smith et al. (2014), Biogeosciences 11, 2027-2054.